In [ ]:
# Scraper for 1996-1999
import requests
from bs4 import BeautifulSoup
import csv
import os
import pandas as pd
import re
import unicodedata
import regex

# Set working directory and output directory
input_csv_path = 
output_dir = 

# Make sure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Read the URLs from the CSV file
urls = pd.read_csv(input_csv_path)['URL'].tolist()  # Assuming the CSV has a column named 'URL'

# Function to normalize and clean text, preserving accents
def normalize_text(text):
    text = unicodedata.normalize('NFKC', text)
    text = regex.sub(r'\p{Mn}+', '', text)
    return text

# Function to add unique institutions, avoiding redundancy
def add_institution(institution_list, institution):
    if institution not in institution_list and not any(institution in item for item in institution_list):
        institution_list.append(institution)
    return institution_list

# Function to clean quote and identify roles and institutions
def clean_quote_and_identify_roles(quote, other_roles, italic_text):
    quote = quote.strip()

    # Detect and remove specific roles
    for role in ['rapporteur', 'author', 'Ombudsman', 'chairman']:
        if re.search(rf'\b{role}\b[.,;]?', italic_text, flags=re.IGNORECASE):
            other_roles = role.capitalize() if not other_roles else f"{other_roles}, {role.capitalize()}"
            quote = re.sub(rf'\b{role}\b[.,;]?', '', quote, flags=re.IGNORECASE).strip()

    return quote, other_roles

# Function to process annex URLs with topic handling
def process_url(url, is_annex=False):
    print(f"Processing {url}...")
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')

    speakers = []
    current_speaker = None
    current_party = None
    current_country = None
    current_form_of_speech = "Spoken"
    current_topic = None  # For annex processing
    current_quote = []
    capturing_quote = False  # Flag to indicate we're collecting quotes for the current speaker or topic

    # Get all <p class="contents"> elements, since both speakers and quotes are inside these
    paragraphs = soup.find_all('p', class_='contents')

    for paragraph in paragraphs:
        # Check if this paragraph contains a speaker (doc_subtitle_level1_bis) or a topic (doc_subtitle_level1_ter)
        speaker_element = paragraph.find('span', class_='doc_subtitle_level1_bis')
        topic_element = paragraph.find('p', class_='doc_subtitle_level1_ter') if is_annex else None

        if speaker_element or topic_element:
            # If there's already a current speaker/topic, save their data before moving to the new one
            if (current_speaker or current_topic) and current_quote:
                # Clean up the quote by removing the speaker, party, and country if they appear
                quote_text = ' '.join(current_quote).strip()
                if current_speaker and current_speaker in quote_text:
                    quote_text = quote_text.replace(current_speaker, '').strip()
                if current_party and current_party in quote_text:
                    quote_text = quote_text.replace(current_party, '').strip()
                if current_country and current_country in quote_text:
                    quote_text = quote_text.replace(current_country, '').strip()
                if current_topic and current_topic in quote_text:
                    quote_text = quote_text.replace(current_topic, '').strip()

                speakers.append({
                    'Speaker': current_speaker,
                    'Party': current_party,
                    'Country': current_country,
                    'Topic': current_topic,  # New column for Topic
                    'Form of Speech': current_form_of_speech,
                    'Quote': quote_text
                })
                current_quote = []  # Reset the quote for the new speaker/topic

            # Now, we are starting to collect data for a new speaker or topic
            if speaker_element:
                current_speaker = normalize_text(speaker_element.text.strip())
                current_topic = None  # Clear the topic when a speaker is found
            elif topic_element and is_annex:
                current_topic = normalize_text(topic_element.text.strip())
                current_speaker = None  # Clear the speaker when a topic is found

            current_party = None
            current_country = None
            capturing_quote = True  # Enable quote capturing for this speaker or topic

            # Extract party from bold text within parentheses (only if speaker is found)
            if speaker_element:
                party_match = paragraph.find('span', class_='bold')
                if party_match and re.match(r'\(.*?\)', party_match.text):
                    current_party = normalize_text(party_match.text[1:-1].strip())

                # Extract country from italic text within parentheses (only if speaker is found)
                country_match = paragraph.find('span', class_='italic')
                if country_match and re.match(r'\(.*?\)', country_match.text):
                    potential_country = normalize_text(country_match.text[1:-1].strip())
                    if potential_country.count(' ') == 0 and potential_country.lower() not in ['applause', 'inaudible', 'heckling']:
                        current_country = potential_country

        # If we're capturing quotes for the current speaker or topic, collect the text
        if capturing_quote:
            # Accumulate the paragraph text, excluding speaker tags, parties, countries, and topics
            quote_text = normalize_text(paragraph.get_text(separator=" ").strip())
            current_quote.append(quote_text)

    # Save the **last speaker or topic's data** if there are any quotes collected
    if (current_speaker or current_topic) and current_quote:
        # Clean up the quote by removing the speaker, party, country, or topic if they appear
        quote_text = ' '.join(current_quote).strip()
        if current_speaker and current_speaker in quote_text:
            quote_text = quote_text.replace(current_speaker, '').strip()
        if current_party and current_party in quote_text:
            quote_text = quote_text.replace(current_party, '').strip()
        if current_country and current_country in quote_text:
            quote_text = quote_text.replace(current_country, '').strip()
        if current_topic and current_topic in quote_text:
            quote_text = quote_text.replace(current_topic, '').strip()

        speakers.append({
            'Speaker': current_speaker,
            'Party': current_party,
            'Country': current_country,
            'Topic': current_topic,  # New column for Topic
            'Form of Speech': current_form_of_speech,
            'Quote': quote_text
        })

    # Determine the file name based on whether it's an annex or regular URL
    file_suffix = "_annex_format" if is_annex else ""
    output_file_path = os.path.join(output_dir, f"{url.split('/')[-1].replace('.html', '')}{file_suffix}.csv")

    # Write the results to a CSV file
    if speakers:
        with open(output_file_path, 'w', encoding='utf-8', newline='') as csv_file:
            fieldnames = ['Speaker', 'Party', 'Country', 'Topic', 'Form of Speech', 'Quote']
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()

            for speaker in speakers:
                writer.writerow(speaker)

        print(f"Results saved to {output_file_path}")
    else:
        print(f"No speakers or topics found for {url}.")

# Process all URLs
for url in urls:
    # Check if it's an annex URL based on the identifier in the URL
    if "ANN-" in url:  # Adjust this condition if needed to identify annex URLs
        process_url(url, is_annex=True)
    else:
        process_url(url, is_annex=False)

print("All URLs processed.")

